In [ ]:
!pip install pyspark

In [1]:
import os
import sys
from config import *

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
collection_uri = MONGO_CONN_URI + "student_erp"
spark_connector = "org.mongodb.spark:mongo-spark-connector_2.12:10.4.0"

#to run pyspark on command line
#(pyspark --packages org.mongodb.spark:mongo-spark-connector_2.12:10.4.0)

In [2]:
from IPython.core.interactiveshell import InteractiveShell


In [3]:
InteractiveShell.ast_node_interactivity = "all"
import pyspark
from pyspark.sql.functions import *
from pyspark.sql import SparkSession, SQLContext, functions as F
from pyspark import SparkContext, SparkConf
from config import *

spark = (SparkSession
         .builder
         .master("local")
         .appName("BDA_Assignment")
         .config("spark.driver.memory", "15g")
         .config("spark.mongodb.read.connection.uri", collection_uri)
         .config("spark.mongodb.write.connection.uri", collection_uri)
         .config("spark.jars.packages", spark_connector)
         .getOrCreate())



In [4]:
import time
execution_times = []

In [5]:
df_students = (spark.read
      .format("mongodb")
      .option("uri", collection_uri)
      .option("database", "student_erp")
      .option("collection", "students")
      .load()
)   

In [6]:
df_departments = (spark.read
      .format("mongodb")
      .option("uri", collection_uri)
      .option("database", "student_erp")
      .option("collection", "departments")
      .load()
)   

In [7]:
df_instructors = (spark.read
      .format("mongodb")
      .option("uri", collection_uri)
      .option("database", "student_erp")
      .option("collection", "instructors")
      .load()
)   

#### QUERY-1:

In [8]:
start_time = time.time()

# Unwind the 'enrollments' array to treat each enrollment as a separate row
df_students_enrollments = df_students.select("student_name", "_id", explode("enrollments").alias("enrollment"))

# Filter by course_id for a specific course
df_students_in_course = df_students_enrollments.filter(df_students_enrollments["enrollment.course_id"] == "CSE101")

# Select relevant columns: student information and course details
df_students_in_course = df_students_in_course.select("student_name", "_id", "enrollment.course_id", "enrollment.course_name")

# Show the result
df_students_in_course.show(truncate=False)

end_time = time.time()
execution_times.append(end_time - start_time)
print(f"Execution Time: {end_time - start_time} seconds")


+-------------+-------+---------+---------------+
|student_name |_id    |course_id|course_name    |
+-------------+-------+---------+---------------+
|Vikram Bansal|BT22029|CSE101   |Data Structures|
|Karthik Reddy|BT22041|CSE101   |Data Structures|
|Praveen Yadav|BT22043|CSE101   |Data Structures|
|Neha Kapoor  |BT24044|CSE101   |Data Structures|
|Tanvi Bansal |BT23058|CSE101   |Data Structures|
|Isha Mehta   |BT22064|CSE101   |Data Structures|
|Kiran Gupta  |BT24063|CSE101   |Data Structures|
|Tanya Desai  |BT22079|CSE101   |Data Structures|
|Priti Iyer   |BT22082|CSE101   |Data Structures|
|Tara Kapoor  |BT23100|CSE101   |Data Structures|
+-------------+-------+---------+---------------+

Execution Time: 8.846636772155762 seconds


#### QUERY-2:

In [9]:
start_time = time.time()

# Unwind the 'enrollments' array to treat each enrollment as a separate row
df_students_enrollments = df_students.select("_id", explode("enrollments").alias("enrollment"))

# Filter by a specific instructor_id
instructor_id = "MEC001"
df_students_instructor = df_students_enrollments.filter(df_students_enrollments["enrollment.instructor.instructor_id"] == instructor_id)

# Group by course_id and count the number of students per course
df_students_per_course = df_students_instructor.groupBy("enrollment.course_id").agg(count("_id").alias("num_students"))

# Calculate the average number of students enrolled in courses taught by the instructor
df_avg_students = df_students_per_course.agg(avg("num_students").alias("avg_students_enrolled"))

# Show the result
df_avg_students.show()

end_time = time.time()
execution_times.append(end_time - start_time)
print(f"Execution Time: {end_time - start_time} seconds")

+---------------------+
|avg_students_enrolled|
+---------------------+
|   25.307692307692307|
+---------------------+

Execution Time: 6.247654676437378 seconds


#### QUERY-3:

In [10]:
start_time = time.time()

department_id = 'CSE'

df_courses_offered = df_departments.select(explode("courses").alias("course"), "_id") \
    .filter(df_departments["_id"] == department_id) \
    .select("course.course_id", "course.course_name", "course.credits", "course.category")
num_rows = df_courses_offered.count()

# Show the courses offered by the specific department
df_courses_offered.show(num_rows, truncate=False)

end_time = time.time()
execution_times.append(end_time - start_time)
print(f"Execution Time: {end_time - start_time} seconds")

+---------+---------------------------+-------+--------+
|course_id|course_name                |credits|category|
+---------+---------------------------+-------+--------+
|CSE101   |Data Structures            |4      |Core    |
|CSE201   |Operating Systems          |4      |Core    |
|CSE202   |Database Systems           |4      |Core    |
|CSE301   |Software Engineering       |4      |Core    |
|CSE302   |Computer Networks          |4      |Core    |
|CSE303   |Web Technologies           |4      |Core    |
|CSE401   |Machine Learning           |4      |Core    |
|CSE402   |Artificial Intelligence    |4      |Core    |
|CSE403   |Mobile App Development     |4      |Core    |
|CSE404   |Cloud Computing            |4      |Elective|
|CSE405   |Cybersecurity              |4      |Elective|
|CSE406   |Ethical Hacking            |2      |Elective|
|CSE407   |Game Development           |2      |Elective|
|CSE408   |Data Visualization         |2      |Elective|
|CSE409   |Human-Computer Inter

#### QUERY-4:

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import count

start_time = time.time()
# Assuming you have already created the Spark session and loaded your DataFrame
# Group by department_id and count the number of students
df_total_students_per_dept = df_students.groupBy("department_id") \
    .agg(count("_id").alias("total_students"))

# Show the result (showing all rows without truncation)
df_total_students_per_dept.show(df_total_students_per_dept.count(), truncate=False)

end_time = time.time()
execution_times.append(end_time - start_time)
print(f"Execution Time: {end_time - start_time} seconds")

+-------------+--------------+
|department_id|total_students|
+-------------+--------------+
|ASH          |39            |
|ENV          |48            |
|ECE          |41            |
|MEC          |40            |
|CIV          |55            |
|CSE          |41            |
|ELE          |36            |
+-------------+--------------+

Execution Time: 1.383951187133789 seconds


#### QUERY-5:

In [12]:
start_time = time.time()

cse_core_courses = df_departments.filter(col("_id") == "CSE") \
    .select("courses") \
    .first()[0]  # Get the list of courses

# Extract course_ids for the core courses
cse_core_course_ids = [course["course_id"] for course in cse_core_courses if course["category"] == "Core"]

# Step 2: Explode enrollments to find instructors who taught those core courses
df_exploded_enrollments = df_students.select(
    col("_id").alias("student_id"),
    explode(col("enrollments")).alias("enrollment"))

df_instructors_taught_courses = df_exploded_enrollments.join(
    df_instructors,
    df_exploded_enrollments.enrollment.instructor.instructor_id == df_instructors._id).filter(col("enrollment.course_id").isin(cse_core_course_ids))

# Step 3: Group by instructor and count distinct courses taught
instructors_with_all_courses = df_instructors_taught_courses.groupBy("instructor_name", "email") \
    .agg(collect_set("enrollment.course_id").alias("courses_taught")) \
    .filter(size(col("courses_taught")) == len(cse_core_course_ids))

# Show the result
instructors_with_all_courses.show(truncate=False)

end_time = time.time()
execution_times.append(end_time - start_time)
print(f"Execution Time: {end_time - start_time} seconds")


+---------------+--------------------------+------------------------------------------------------------------------+
|instructor_name|email                     |courses_taught                                                          |
+---------------+--------------------------+------------------------------------------------------------------------+
|Dr. Priya Desai|priya.desai@university.edu|[CSE202, CSE303, CSE302, CSE301, CSE201, CSE402, CSE101, CSE401, CSE403]|
+---------------+--------------------------+------------------------------------------------------------------------+

Execution Time: 2.4381191730499268 seconds


#### QUERY-6:

In [13]:
start_time = time.time()

# Unwind the 'enrollments' array to treat each enrollment as a separate row
df_students_enrollments = df_students.select(explode("enrollments").alias("enrollment"))

# Group by course_id and count the number of enrollments for each course
df_course_enrollments = df_students_enrollments.groupBy("enrollment.course_id") \
    .agg(count("enrollment.course_id").alias("enrollment_count"))

# Sort by enrollment count in descending order and limit to the top 10 courses
df_top_10_courses = df_course_enrollments.orderBy("enrollment_count", ascending=False).limit(10)

# Show the result
df_top_10_courses.show()

end_time = time.time()
execution_times.append(end_time - start_time)
print(f"Execution Time: {end_time - start_time} seconds")

+---------+----------------+
|course_id|enrollment_count|
+---------+----------------+
|   ECE304|              64|
|   ECE303|              60|
|   ECE301|              57|
|   ENV205|              53|
|   ECE201|              51|
|   ECE101|              49|
|   ASH201|              49|
|   ECE202|              49|
|   ENV204|              49|
|   ENV202|              48|
+---------+----------------+

Execution Time: 1.3667304515838623 seconds


In [14]:
execution_times

[8.846636772155762,
 6.247654676437378,
 1.0456454753875732,
 1.383951187133789,
 2.4381191730499268,
 1.3667304515838623]